In [ ]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
from tqdm import tqdm
#sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
import torch.nn.functional as F
import math
from typing import Optional
from tsnn.tstorch import transformers
from sklearn.linear_model import LinearRegression, ElasticNetCV, ElasticNet




plt.style.use('ggplot')

In [ ]:
from dataclasses import dataclass
from torch import nn


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

In [ ]:
from typing import Dict

In [ ]:
from tsnn.tstorch import models

In [ ]:
from tsnn.tstorch.models import GlobalMLP, BiDimensionalMLP, OneDimensionalTransformer, CustomBiDimensionalTransformer
from sklearn.ensemble import HistGradientBoostingRegressor



# Summary

In [ ]:
# In this notebook we will run the experiments to generate the figures for the paper.

# Test dataset

In [ ]:
# We work with the following data.

In [ ]:
# Global parameters
T_max = 5000
N1 = 10
F1 = 20
T1 = 5 # This parameter will be the n_rolling

print(T_max, N1, F1, T1)

In [ ]:
def generate_synthetic_datasets(
    num_time_steps: int = 3000,
    num_time_series: int = 10,
    num_features: int = 10,
    low_corr: float = 0.1,
    high_corr: float = 0.2,
    pct_zero_corr: float = 0.5,
) -> Dict[str, "generators.Generator"]:
    """
    Generates 5 synthetic multivariate time series datasets with different
    types of cross-series dependencies.

    Returns
    -------
    dict
        Keys: "d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"
        Values: generators.Generator objects (already with .train and .test)
    """
    dic_data = {}

    # Helper to avoid repeating the same 10 lines
    def make_gen(split_conditional=0.0,
                 split_shift=0.0,
                 split_seasonal=0.0,
                 split_cs=0.0,
                 split_cs_shift=0.0):
        gen = generators.Generator(num_time_steps, num_time_series, num_features)
        gen.generate_dataset(
            pct_zero_corr=pct_zero_corr,
            split_conditional=split_conditional,
            split_shift=split_shift,
            split_seasonal=split_seasonal,
            split_cs=split_cs,
            split_cs_shift=split_cs_shift,
            low_corr=low_corr,
            high_corr=high_corr,
        )
        return gen

    dic_data["d_lin"] = make_gen()

    # 1. Pure conditional (causal) dependence
    dic_data["d_cond"] = make_gen(split_conditional=1.0)

    # 2. Pure lagged (time-shifted) dependence
    dic_data["d_shift"] = make_gen(split_shift=1.0)

    # 3. Pure contemporaneous cross-sectional correlation
    dic_data["d_cs"] = make_gen(split_cs=1.0)

    # 4. Contemporaneous + lagged cross-series
    dic_data["d_cs_shift"] = make_gen(split_cs_shift=1.0)

    # 5. Equal mix of all four mechanisms
    dic_data["d_all"] = make_gen(
        split_conditional=0.2,
        split_shift=0.2,
        split_cs=0.2,
        split_cs_shift=0.2,
    )

    return dic_data

In [ ]:
# list_low_corr = [0.01, 0.025, 0.05, 0.1]
# list_high_corr = [2*x for x in list_low_corr]

list_low_corr = [0.01, 0.03, 0.05, 0.1, 0.3]
list_high_corr = list_low_corr

dic_data = {}

for i in range(len(list_low_corr)):
    name = "correl" + str(list_low_corr[i])
    dic_data[name] = generate_synthetic_datasets(num_time_steps=T_max, num_time_series=N1, num_features=F1, low_corr=list_low_corr[i], high_corr=list_high_corr[i])
    

In [ ]:
# We will fix the above dataset for now.

In [ ]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

mask = causal_mask
mask = build_attention_mask(mask, T1, device=device)
def custom_mask_mod(b, h, q_idx, kv_idx):
    return mask[q_idx, kv_idx]

# List of models

In [ ]:
# Let's list here all the models we wish to test on all the data.

In [ ]:
def get_models():
    MLP_global = GlobalMLP(N1, F1, T1, dropout=0.2).to(device)

    MLP_2D = BiDimensionalMLP(N1, F1, T1, dropout=0.2).to(device)

    trans_1D_T4 = OneDimensionalTransformer(N1, F1, T1, mask=mask, attn_direction="T",  num_attn_layers=4,
                                            dropout=0.2, roll_y=True).to(device)
    #Note: using the MLP compression seems very bad..

    trans_2D_TCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)

    trans_2D_TCTCTCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)


    dic_models = {'MLP_global':MLP_global, 'MLP_2D':MLP_2D, "trans_1D_T4":trans_1D_T4, "trans_2D_TCTC":trans_2D_TCTC, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC}

    trans_2D_TCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)

    trans_2D_TCTCTCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)


    dic_models_rollfalse = {"trans_2D_TCTC":trans_2D_TCTC_rollfalse, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC_rollfalse}
    
    return dic_models, dic_models_rollfalse

In [ ]:
# For each model we also need to specify the option we will use to fit them.

# Function to create table for an effect

In [ ]:
# We give the function that creates for a given effect the table testing all models and all noise level.

In [ ]:
def run_sim(model, n=200, N=N1*T1*F1, T=T_max, rho=0, oos=True):
    res = 0
    for k in tqdm(range(n)):
        
        # constant rho_i case
        rho_i = np.array([rho * np.sqrt(1 / N) for k in range(N)])

        # random rho_i re-normalized
        rho_i = np.random.uniform(low=-1, high=1, size=N)
        rho_i = np.random.normal(size=N)
        norm_factor = np.sum([rho_i[k]**2 for k in range(N)])
        rho_i = np.sqrt(np.abs(rho_i / norm_factor)) * np.sign(rho_i)
        rho_i = np.sqrt((rho_i**2) * (rho**2)) * np.sign(rho_i)


        mean = np.zeros(N)
        cov = np.eye(N)
        X = np.random.multivariate_normal(mean=mean, cov=cov, size=T)
        eps = np.random.normal(size=T)

        y_tilde = np.dot(X, rho_i)
        y = y_tilde + eps

        model.fit(X[:int(len(X)/2)], y[:int(len(X)/2)])
        if oos:
            res += np.corrcoef(model.predict(X[int(len(X)/2):]), y_tilde[int(len(X)/2):])[0][1]
        else:
            res += np.corrcoef(model.predict(X[:int(len(X)/2)]), y_tilde[:int(len(X)/2):])[0][1]
    return res / n

def get_ols_corr( N=N1*T1*F1, T=T_max, rho=0):
    return rho / np.sqrt(rho**2 + (1-rho**2) * min(N, T) / T)


def run_models(effect1):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in list(dic_data.keys()):
        
        z = dic_data[noise_level][effect1]
        z.get_dataloader(n_rolling=T1)

        dic_models, dic_models_rollfalse = get_models()


        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        records_train.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        records_train.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })
        


        for model_key in dic_models.keys():                   
            print(f"Running → {noise_level} | {model_key}")

            z = dic_data[noise_level][effect1]
            if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1, roll_y=True)
            else:
                z.get_dataloader(n_rolling=T1)

            if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                lr=0.001/2
            else:
                lr=0.001

            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40

            model = dic_models[model_key]

            # Model
            if noise_level == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1)
                model = dic_models_rollfalse[model_key]
                epochs = 60
                lr=0.0001

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

            
            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "model": model_key,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "model": model_key,
                "test_corr_optimal": test_corr
            })

        records_train.append({
                    "noise_level": noise_level,
                    "model": 'max_linear_corr',
                    # "train_corr_optimal": run_sim(ElasticNetCV(l1_ratio=1, alphas=np.linspace(1e-6, 1e-2, num=50), cv=3), N=N1*F1*T1, 
                    #                               T=T_max, rho=float(noise_level[6:]), oos=False)
                    'train_corr_optimal': get_ols_corr(N=N1*F1*T1, T=T_max, rho=np.sqrt(0.5*F1*float(noise_level[6:])**2))

                })

        records_test.append({
                    "noise_level": noise_level,
                    "model": 'max_linear_corr',
                    # "test_corr_optimal": run_sim(ElasticNetCV(l1_ratio=1, alphas=np.linspace(1e-6, 1e-2, num=50), cv=3), N=N1*F1*T1, 
                    #                              T=T_max, rho=float(noise_level[6:]), oos=True)
                    'test_corr_optimal': get_ols_corr(N=N1*F1*T1, T=T_max, rho=np.sqrt(0.5*F1*float(noise_level[6:])**2))
                })
    
    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="model", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="model", values="test_corr_optimal")

    col_order = ["lasso_full", "boosting"] + list(dic_models.keys()) + ['max_linear_corr']
    train_pivot = train_pivot[col_order]
    test_pivot  = test_pivot[col_order]

    return train_pivot, test_pivot

## Running on linear effect

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_lin0, table_test_lin0 = run_models('d_lin')

In [ ]:
display(table_train_lin0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_lin0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
# Saving the data

table_train_lin0.to_csv('table_train_lin.csv', index=True)
table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on conditional effect

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_cond0, table_test_cond0 = run_models('d_cond')

In [ ]:
display(table_train_cond0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_cond0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
# Saving the data

#table_train_cond0.to_csv('table_train_cond.csv', index=True)
#table_test_cond0.to_csv('table_test_cond.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on shift effect

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_shift0, table_test_shift0 = run_models('d_shift')

In [ ]:
display(table_train_shift0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_shift0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
# Saving the data

table_train_shift0.to_csv('table_train_shift.csv', index=True)
table_test_shift0.to_csv('table_test_shift.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs effect

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_cs0, table_test_cs0 = run_models('d_cs')

In [ ]:
display(table_train_cs0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_cs0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
# Saving the data

table_train_cs0.to_csv('table_train_cs.csv', index=True)
table_test_cs0.to_csv('table_test_cs.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs_shift

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_csshift0, table_test_csshift0 = run_models('d_cs_shift')

In [ ]:
display(table_train_csshift0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))
display(table_test_csshift0.T.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
# Saving the data

table_train_csshift0.to_csv('table_train_csshift.csv', index=True)
table_test_csshift0.to_csv('table_test_csshift.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

# Function to create table for all effects

In [ ]:
def run_models_all_effect(noise_level1, dic_models, dic_models_rollfalse):

    z = dic_data[noise_level1]["d_all"]
    z.get_dataloader(n_rolling=T1)

    lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                            verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
    lasso_full.fit(z.train)
    boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
    boost_model.fit(z.train)

    list_models = [lasso_full, boost_model]


    for model_key in dic_models.keys():                   
        print(f"Running → {noise_level1} | {model_key}")

        if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1, roll_y=True)
        else:
            z.get_dataloader(n_rolling=T1)

        if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            lr=0.001/2
        else:
            lr=0.001

        epochs = 20
        if noise_level1 in ['correl0.01', 'correl0.03']:
            epochs = 40

        model = dic_models[model_key]

        # Model
        if noise_level1 == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1)
            model = dic_models_rollfalse[model_key]
            epochs = 60
            lr=0.0001

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

        
        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

        list_models.append(wrapper)

    comp = benchmark_comparison.Comparator(models=list_models, model_names=["lasso_full", "boosting"] + list(dic_models.keys()))

    corr_train = comp.correl(z, mode="train", return_values=True)
    corr_test  = comp.correl(z, mode="test",  return_values=True)

    return corr_train, corr_test

    

## Run on all noise levels

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_1, all_effects_test_1 = run_models_all_effect("correl0.01", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_3, all_effects_test_3 = run_models_all_effect("correl0.03", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_5, all_effects_test_5 = run_models_all_effect("correl0.05", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_10, all_effects_test_10 = run_models_all_effect("correl0.1", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_30, all_effects_test_30 = run_models_all_effect("correl0.3", dic_models, dic_models_rollfalse)

In [ ]:
display(all_effects_train_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_test_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_train_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_test_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_train_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_test_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_train_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_test_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_train_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(all_effects_test_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
# Let's also save all the tables above.
all_effects_train_1.to_csv('all_effects_train_1.csv', index=True)
all_effects_test_1.to_csv('all_effects_test_1.csv', index=True)

all_effects_train_3.to_csv('all_effects_train_3.csv', index=True)
all_effects_test_3.to_csv('all_effects_test_3.csv', index=True)

all_effects_train_5.to_csv('all_effects_train_5.csv', index=True)
all_effects_test_5.to_csv('all_effects_test_5.csv', index=True)

all_effects_train_10.to_csv('all_effects_train_10.csv', index=True)
all_effects_test_10.to_csv('all_effects_test_10.csv', index=True)

all_effects_train_30.to_csv('all_effects_train_30.csv', index=True)
all_effects_test_30.to_csv('all_effects_test_30.csv', index=True)

# Testing sparsity

In [ ]:
# To test sparsity let's write a function that runs one model on all the effects and noise levels.
# Then we can run on the model with and without sparsity.

In [ ]:
def keep_topk_per_row(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out

In [ ]:
def test_sparsity_all_correl_effets(sparsity=False):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():          
        for effect in ['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']:                   
            print(f"Running → {noise_level} | {effect}")

            z = dic_data[noise_level][effect]
            z.get_dataloader(n_rolling=T1, roll_y=True)
            
            lr=0.001/2
            roll_y=True
            
            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40


            # Model
            if noise_level == 'correl0.01':
                z.get_dataloader(n_rolling=T1)
                roll_y=False
                epochs = 60
                lr=0.0001

            # Model
            if sparsity:
                model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=keep_topk_per_row, 
                                                                               roll_y=roll_y).to(device)
            else:
                model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                                roll_y=roll_y).to(device)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "effect": effect,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "effect": effect,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="effect", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="effect", values="test_corr_optimal")    

    # Sort columns in logical order
    col_order = ["d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"]
    train_model = train_pivot[col_order]
    test_model  = test_pivot[col_order]

    return train_model, test_model

In [ ]:
table_train_no_sparsity, table_test_no_sparsity = test_sparsity_all_correl_effets()

In [ ]:
table_train_with_sparsity, table_test_with_sparsity = test_sparsity_all_correl_effets(sparsity=True)

In [ ]:
display(table_train_no_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(table_train_with_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(table_test_no_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(table_test_with_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
# To store the tables above

table_train_no_sparsity.to_csv('table_train_no_sparsity.csv', index=True)
table_test_no_sparsity.to_csv('table_test_no_sparsity.csv', index=True)

table_train_with_sparsity.to_csv('table_train_with_sparsity.csv', index=True)
table_test_with_sparsity.to_csv('table_test_with_sparsity.csv', index=True)

### Sparsity boostrap

In [ ]:
dic_data

In [ ]:
def bootstrap_sparsity(noise_level='correl0.03', effect='d_all', n=10):

    # Storage
    records_train = []
    records_test  = []               
    print(f"Running → {noise_level} | {effect}")

    z = dic_data[noise_level][effect]
    z.get_dataloader(n_rolling=T1, roll_y=True)

    lr=0.001/2
    roll_y=True

    epochs = 20
    if noise_level in ['correl0.01', 'correl0.03']:
        epochs = 40

    for k in tqdm(range(n)):
        # Model
        if noise_level == 'correl0.01':
            z.get_dataloader(n_rolling=T1)
            roll_y=False
            epochs = 60
            lr=0.0001

        model_sparse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=keep_topk_per_row, 
                                                                           roll_y=roll_y).to(device)

        model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                            roll_y=roll_y).to(device)

        ##################### not sparse
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)

        # Compare
        comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        train_corr = corr_train.loc["model1", "optimal"]
        test_corr  = corr_test.loc["model1", "optimal"]
        
        ###################### sparse
        optimizer = torch.optim.Adam(model_sparse.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model_sparse, optimizer=optimizer, device=device)

        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False, verbose=0)

        # Compare
        comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

        corr_train_sparse = comp.correl(z, mode="train", return_values=True)
        corr_test_sparse  = comp.correl(z, mode="test",  return_values=True)

        train_corr_sparse = corr_train_sparse.loc["model1", "optimal"]
        test_corr_sparse  = corr_test_sparse.loc["model1", "optimal"]

        # Save results
        records_train.append({
            "train_corr_sparse": train_corr_sparse,
            "train_corr": train_corr
        })
        records_test.append({
            "test_corr_sparse": test_corr_sparse,
            "test_corr": test_corr
        })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    return df_train, df_test

#### Correl 0.01

In [ ]:
for effect in dic_data['correl0.01']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.01', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))

#### Correl 0.03

In [ ]:
for effect in dic_data['correl0.03']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.03', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))

#### Correl 0.05

In [ ]:
for effect in dic_data['correl0.05']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.05', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))

#### Correl 0.1

In [ ]:
for effect in dic_data['correl0.1']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.1', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))

#### Correl 0.3

In [ ]:
for effect in dic_data['correl0.3']:
    table_train, table_test = bootstrap_sparsity(noise_level='correl0.3', effect=effect, n=40)
    display(pd.concat([table_test.quantile([0.1, 0.9]), table_test.aggregate(['mean', 'std'])]).T.rename_axis(effect, axis=0).style.background_gradient().format(precision=4))